[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/27_deep_learning_foundations/90_exercises.ipynb)

# Bài tập — Chương 27: Deep Learning Foundations (L0 → L7)

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/27_deep_learning_foundations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Bài tập này cần PyTorch: pip install -e \".[dl]\"") from e

rng = np.random.default_rng(0)
_ = torch.manual_seed(0)

## L0 — Nhớ khái niệm

**Câu 1.** (a) Một nơ-ron với kích hoạt sigmoid tương đương mô hình nào? (b) Quy tắc học perceptron; điều kiện hội tụ. (c) Quy tắc chuỗi trong backprop; vì sao gradient phải **cộng dồn** khi một biến dùng ở nhiều nơi? (d) Gradient của softmax + cross-entropy theo logit. (e) 5 bước của vòng lặp huấn luyện PyTorch.

✍️ Trả lời:

**Đáp án:**
(a) Hồi quy logistic: $\sigma(\mathbf w^\top\mathbf x + b)$, ranh giới tuyến tính.
(b) Khi đoán sai điểm $(\mathbf x, y)$, $y \in \{\pm1\}$: $\mathbf w \leftarrow \mathbf w + \eta y\mathbf x$, $b \leftarrow b + \eta y$. Hội tụ (≤ $(R/\gamma)^2$ lỗi) nếu dữ liệu tách được tuyến tính với lề $\gamma > 0$.
(c) $\partial L/\partial u = \partial L/\partial v \cdot \partial v/\partial u$ — gradient phía sau × gradient cục bộ. Biến ảnh hưởng tới $L$ qua nhiều đường → đạo hàm toàn phần là **tổng** đóng góp của mọi đường.
(d) $\partial L/\partial\mathbf z = \hat{\mathbf p} - \mathbf y$ (one-hot), chia $n$ nếu lấy trung bình.
(e) `zero_grad` → forward → loss → `backward` → `step`.

## L1 — Giải thích bằng lời

**Câu 2.** Một bạn nói: *"Theo định lý xấp xỉ vạn năng, mạng một lớp ẩn với 10 000 nơ-ron học được mọi thứ, nên không cần mạng sâu hay thiết kế kiến trúc."* Phản biện.

✍️ Trả lời:

**Gợi ý:** Định lý chỉ khẳng định **tồn tại** bộ trọng số xấp xỉ tốt trên miền bị chặn; không nói (1) gradient descent **tìm được** chúng, (2) cần **bao nhiêu** nơ-ron — với nhiều hàm, mạng nông cần số nơ-ron tăng theo hàm mũ trong khi mạng sâu chỉ cần đa thức (tái sử dụng đặc trưng theo tầng), (3) mô hình **tổng quát hoá** ngoài dữ liệu train — 10 000 nơ-ron với ít dữ liệu dễ thuộc lòng. Kiến trúc mã hoá **giả định** phù hợp dữ liệu (tích chập cho ảnh, tuần tự/attention cho chuỗi — chương 29–31) giúp học với ít dữ liệu và tính toán hơn nhiều.

## L2 — Tính tay

**Câu 3.** Mạng: $\mathbf x = [1, 2]$, $W_1 = \begin{bmatrix}0.5 & -1\\1 & 0.5\end{bmatrix}$ (cột = nơ-ron ẩn), $\mathbf b_1 = \mathbf 0$, ReLU, $\mathbf w_2 = [1, -2]$, $b_2 = 0.5$, đầu ra tuyến tính $\hat y$; loss $L = \frac12(\hat y - y)^2$ với $y = 1$. Tính $\hat y$, $L$, $\partial L/\partial\mathbf w_2$, $\partial L/\partial b_2$, $\partial L/\partial W_1$.

In [ ]:
x3 = np.array([1.0, 2.0])
W1_3 = np.array([[0.5, -1.0], [1.0, 0.5]])
w2_3, b2_3, y3 = np.array([1.0, -2.0]), 0.5, 1.0
### START CODE HERE ### (≈ 9 dòng)
z1 = x3 @ W1_3
h = np.maximum(0, z1)
y_hat3 = h @ w2_3 + b2_3
L3 = 0.5 * (y_hat3 - y3) ** 2
dy = y_hat3 - y3
dw2 = dy * h
db2 = dy
dz1 = dy * w2_3 * (z1 > 0)
dW1 = np.outer(x3, dz1)
### END CODE HERE ###
print(f"ŷ = {y_hat3}, L = {L3}, dL/dw2 = {dw2}, dL/db2 = {db2}\ndL/dW1 =\n{dW1}")

In [ ]:
check_close(y_hat3, 3.0, message="ŷ")
check_close(L3, 2.0, message="loss")
check_close(dw2, [5.0, 0.0], message="dL/dw2")
check_close(db2, 2.0, message="dL/db2")
check_close(dW1, [[2.0, 0.0], [4.0, 0.0]], message="dL/dW1 (nơ-ron thứ 2 có z = 0 → ReLU chặn gradient)")
tW1 = torch.tensor(W1_3, requires_grad=True)
tl = 0.5 * ((torch.relu(torch.tensor(x3) @ tW1) @ torch.tensor(w2_3) + b2_3 - y3) ** 2)
tl.backward()
check_close(dW1, tW1.grad.numpy(), message="khớp torch.autograd")

**Câu 4.** Số tham số của mạng fully-connected 784 → 256 → 128 → 10 (có bias).

In [ ]:
### START CODE HERE ### (≈ 1 dòng)
n_params4 = 784 * 256 + 256 + 256 * 128 + 128 + 128 * 10 + 10
### END CODE HERE ###
print(n_params4)

In [ ]:
model4 = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))
check_equal(n_params4, sum(p.numel() for p in model4.parameters()), "khớp PyTorch")

## L3 — Code

**Bài 5.** Cài đặt các hàm thuần NumPy:
- `linear_forward(X, W, b)` → `Z`; `linear_backward(dZ, X, W)` → `(dX, dW, db)`;
- `softmax_ce_forward(Z, y)` → `(loss, probs)` (ổn định số, loss trung bình); `softmax_ce_backward(probs, y)` → `dZ`.

Kiểm tra với `torch.autograd` trên dữ liệu ngẫu nhiên.

In [ ]:
def linear_forward(X, W, b):
    ### START CODE HERE ### (≈ 1 dòng)
    # STARTER: return None
    return X @ W + b
    ### END CODE HERE ###


def linear_backward(dZ, X, W):
    ### START CODE HERE ### (≈ 1 dòng)
    # STARTER: return None, None, None   # dX, dW, db
    return dZ @ W.T, X.T @ dZ, dZ.sum(axis=0)
    ### END CODE HERE ###


def softmax_ce_forward(Z, y):
    ### START CODE HERE ### (≈ 4 dòng)
    # STARTER: return None, None   # loss trung bình, probs — nhớ trừ max theo hàng trước exp
    S = Z - Z.max(axis=1, keepdims=True)
    log_p = S - np.log(np.exp(S).sum(axis=1, keepdims=True))
    return -log_p[np.arange(len(y)), y].mean(), np.exp(log_p)
    ### END CODE HERE ###


def softmax_ce_backward(probs, y):
    ### START CODE HERE ### (≈ 3 dòng)
    # STARTER: return None   # (probs − one_hot(y)) / n
    d = probs.copy()
    d[np.arange(len(y)), y] -= 1
    return d / len(y)
    ### END CODE HERE ###

In [ ]:
X5, W5, b5 = rng.normal(size=(6, 4)), rng.normal(size=(4, 3)), rng.normal(size=3)
y5 = rng.integers(0, 3, 6)
Z5 = linear_forward(X5, W5, b5)
loss5, P5 = softmax_ce_forward(Z5, y5)
dX5, dW5, db5 = linear_backward(softmax_ce_backward(P5, y5), X5, W5)
tX, tW, tb = (torch.tensor(a, requires_grad=True) for a in (X5, W5, b5))
tloss = nn.functional.cross_entropy(tX @ tW + tb, torch.tensor(y5))
tloss.backward()
check_close(loss5, tloss.item(), tol=1e-10, message="loss")
check_close(dW5, tW.grad.numpy(), tol=1e-10, message="dW")
check_close(db5, tb.grad.numpy(), tol=1e-10, message="db")
check_close(dX5, tX.grad.numpy(), tol=1e-10, message="dX")
big_loss, _ = softmax_ce_forward(np.array([[1000.0, 0.0]]), np.array([0]))
check_true(np.isfinite(big_loss), "ổn định số với logit rất lớn")

**Bài 6.** Cài đặt `perceptron_fit(X, y, max_epochs)` (nhãn ±1, $\eta = 1$, duyệt theo thứ tự cố định) trả về `(w, b, n_mistakes)`. Trên dữ liệu tách được với lề ≥ $\gamma$ quanh siêu phẳng $x_1 - x_2 = 0$, kiểm tra số lỗi tổng cộng không vượt cận Novikoff $(R/\gamma)^2$ (với dữ liệu mở rộng thêm cột 1 cho bias: $R = \max\lVert[\mathbf x, 1]\rVert$, và lề của siêu phẳng đi qua gốc trong không gian mở rộng).

In [ ]:
def perceptron_fit(X, y, max_epochs=100):
    w, b, mistakes = np.zeros(X.shape[1]), 0.0, 0
    ### START CODE HERE ### (≈ 9 dòng)
    # STARTER: # lặp epoch; với từng i: nếu y[i]·(w·x_i + b) ≤ 0 → w += y[i]·x_i, b += y[i], mistakes += 1
    # STARTER: # dừng sớm khi một epoch không có lỗi
    for _ in range(max_epochs):
        errors = 0
        for i in range(len(X)):
            if y[i] * (X[i] @ w + b) <= 0:
                w += y[i] * X[i]
                b += y[i]
                errors += 1
        mistakes += errors
        if errors == 0:
            break
    ### END CODE HERE ###
    return w, b, mistakes

In [ ]:
X6 = rng.uniform(-1, 1, (2000, 2))
margin6 = 0.15
w_star = np.array([1.0, -1.0]) / np.sqrt(2)
keep = np.abs(X6 @ w_star) >= margin6
X6, y6 = X6[keep][:300], np.sign(X6[keep][:300] @ w_star)
w6, b6, mistakes6 = perceptron_fit(X6, y6)
R = np.max(np.linalg.norm(np.c_[X6, np.ones(len(X6))], axis=1))
gamma = np.min(y6 * (X6 @ w_star))                                      # lề thật của w* (b* = 0) — trong không gian mở rộng w* có thêm thành phần 0
bound = (R / gamma) ** 2
print(f"số lỗi = {mistakes6} | cận (R/γ)² = {bound:.1f} | accuracy = {np.mean(np.sign(X6 @ w6 + b6) == y6):.3f}")
check_equal(float(np.mean(np.sign(X6 @ w6 + b6) == y6)), 1.0, "tách hoàn hảo")
check_less(mistakes6, bound, "số lỗi không vượt cận Novikoff")

## L4 — Debug

**Bài 7.** Vòng huấn luyện PyTorch dưới đây (MLP cho digits) có 3 lỗi: (1) quên `optimizer.zero_grad()`; (2) mô hình áp **softmax** rồi mới đưa vào `CrossEntropyLoss`; (3) báo cáo accuracy trên **tập train**. Sửa và lưu `test_acc7` (accuracy trên `X7_te`).

In [ ]:
X7, y7 = load_digits(return_X_y=True)
X7_tr, X7_te, y7_tr, y7_te = train_test_split(X7, y7, test_size=0.25, random_state=0, stratify=y7)
sc7 = StandardScaler().fit(X7_tr)
Xt_tr, Xt_te = (torch.tensor(sc7.transform(a), dtype=torch.float32) for a in (X7_tr, X7_te))
yt_tr, yt_te = torch.tensor(y7_tr), torch.tensor(y7_te)
torch.manual_seed(0)
### START CODE HERE ### (≈ 14 dòng)
# STARTER: model7 = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10), nn.Softmax(dim=1))   # 🐛 (2)
# STARTER: opt7 = torch.optim.SGD(model7.parameters(), lr=0.1)
# STARTER: loss_fn7 = nn.CrossEntropyLoss()
# STARTER: for epoch in range(30):
# STARTER:     perm = torch.randperm(len(Xt_tr))
# STARTER:     for s in range(0, len(perm), 32):
# STARTER:         idx = perm[s:s + 32]
# STARTER:         loss = loss_fn7(model7(Xt_tr[idx]), yt_tr[idx])                                          # 🐛 (1): thiếu zero_grad
# STARTER:         loss.backward()
# STARTER:         opt7.step()
# STARTER: with torch.no_grad():
# STARTER:     test_acc7 = (model7(Xt_tr).argmax(1) == yt_tr).float().mean().item()                       # 🐛 (3)
model7 = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))
opt7 = torch.optim.SGD(model7.parameters(), lr=0.1)
loss_fn7 = nn.CrossEntropyLoss()
for epoch in range(30):
    perm = torch.randperm(len(Xt_tr))
    for s in range(0, len(perm), 32):
        idx = perm[s:s + 32]
        opt7.zero_grad()
        loss = loss_fn7(model7(Xt_tr[idx]), yt_tr[idx])
        loss.backward()
        opt7.step()
with torch.no_grad():
    test_acc7 = (model7(Xt_te).argmax(1) == yt_te).float().mean().item()
### END CODE HERE ###
print(f"accuracy test = {test_acc7:.4f}")

In [ ]:
check_greater(test_acc7, 0.95, "accuracy test > 0.95")
check_true(not any(isinstance(m, nn.Softmax) for m in model7.modules()), "không có lớp Softmax trước CrossEntropyLoss")

## L5 — Phân tích

**Bài 8.** **Learning rate.** Huấn luyện MLP NumPy (2 → 64 → 3, ReLU — dùng `linear_forward/backward`, `softmax_ce_*` của Bài 5) bằng gradient descent **toàn batch** 300 bước trên dữ liệu xoắn ốc với $\eta \in \{0.001, 0.01, 0.1, 1, 10\}$. Lưu `curves8[lr]` = list loss theo bước (dừng sớm, gán `np.inf`, nếu loss thành NaN/inf). Nhận xét.

In [ ]:
def make_spiral(n_per_class=100, n_classes=3, noise=0.2, seed=0):
    r = np.random.default_rng(seed)
    X, y = [], []
    for k in range(n_classes):
        t = np.linspace(0, 1, n_per_class)
        theta = np.linspace(k * 4, (k + 1) * 4, n_per_class) + r.normal(0, noise, n_per_class)
        X.append(np.c_[t * np.sin(theta), t * np.cos(theta)])
        y.append(np.full(n_per_class, k))
    return np.vstack(X), np.concatenate(y)


X8, y8 = make_spiral()
curves8 = {}
### START CODE HERE ### (≈ 18 dòng)
# STARTER: # với mỗi lr: khởi tạo W1 ~ N(0, √(2/2)), b1 = 0, W2 ~ N(0, √(2/64)), b2 = 0 (cùng seed 0 cho mọi lr)
# STARTER: # 300 bước: forward (linear → ReLU → linear → softmax-CE), backward, cập nhật; ghi loss; dừng nếu không hữu hạn
for lr in [0.001, 0.01, 0.1, 1.0, 10.0]:
    r8 = np.random.default_rng(0)
    W1, b1 = r8.normal(0, 1.0, (2, 64)), np.zeros(64)
    W2, b2 = r8.normal(0, np.sqrt(2 / 64), (64, 3)), np.zeros(3)
    losses = []
    for _ in range(300):
        Z1 = linear_forward(X8, W1, b1)
        H1 = np.maximum(0, Z1)
        Z2 = linear_forward(H1, W2, b2)
        with np.errstate(over="ignore", invalid="ignore"):
            loss, P = softmax_ce_forward(Z2, y8)
        if not np.isfinite(loss):
            losses.append(np.inf)
            break
        losses.append(loss)
        dH1, dW2, db2 = linear_backward(softmax_ce_backward(P, y8), H1, W2)
        _, dW1, db1 = linear_backward(dH1 * (Z1 > 0), X8, W1)
        W1, b1, W2, b2 = W1 - lr * dW1, b1 - lr * db1, W2 - lr * dW2, b2 - lr * db2
    curves8[lr] = losses
### END CODE HERE ###
for lr, c in curves8.items():
    print(f"lr = {lr:<6}: loss đầu {c[0]:.3f} → cuối {c[-1]:.3f} ({len(c)} bước)")
plt.figure(figsize=(9, 4))
for lr, c in curves8.items():
    plt.plot(np.clip(c, 0, 3), label=f"lr = {lr}")
plt.legend()
plt.xlabel("bước")
plt.ylabel("loss (cắt ở 3)")
plt.show()

In [ ]:
final = {lr: c[-1] for lr, c in curves8.items()}
check_greater(final[0.001], 0.9, "lr quá nhỏ: hầu như chưa học")
check_less(final[1.0], final[0.01], "lr vừa phải học nhanh hơn nhiều")
check_true(not np.isfinite(final[10.0]) or final[10.0] > final[1.0], "lr quá lớn: phân kỳ hoặc tệ hơn")

**Nhận xét mẫu:** $\eta$ quá nhỏ (0.001): loss gần như không đổi sau 300 bước — lãng phí tính toán. $\eta$ vừa (0.1–1): giảm nhanh và ổn định. $\eta$ quá lớn (10): mỗi bước "nhảy" qua đáy rồi phóng đại sai số → loss dao động ở mức **cao hơn cả lúc khởi tạo** (lớn hơn nữa thì thành NaN). Learning rate là siêu tham số quan trọng nhất; thực tế dùng **LR range test** (tăng dần LR, chọn trước điểm loss bắt đầu bùng nổ), lịch giảm LR và optimizer thích nghi (Adam) — chương 28.

## L6 — Thực tế

**Bài 9.** **Mạng nơ-ron cho dữ liệu bảng.** Dữ liệu `customer_churn.csv`. Chia train/test 75/25 (`random_state=0`, stratify). Baseline: `LogisticRegression` trên cột số chuẩn hoá + one-hot (dùng `ColumnTransformer`) → `auc_lr`. Mạng PyTorch (≥ 1 lớp ẩn, dùng **cùng** tiền xử lý đã fit trên train) huấn luyện bằng `BCEWithLogitsLoss` + Adam, early stopping theo validation tách từ train → `auc_mlp` trên test. Yêu cầu: `auc_mlp ≥ auc_lr − 0.01`. Nhận xét: mạng có đáng không?

In [ ]:
churn = pd.read_csv(repo_path("data", "sample", "customer_churn.csv"))
y9 = churn["churn"].to_numpy()
X9 = churn.drop(columns=["customer_id", "churn"])
num_cols = ["tenure_months", "monthly_charges", "total_charges", "support_calls", "n_products"]
cat_cols = ["contract", "internet_service", "payment_method"]
X9_tr, X9_te, y9_tr, y9_te = train_test_split(X9, y9, test_size=0.25, random_state=0, stratify=y9)
pre9 = ColumnTransformer([("num", StandardScaler(), num_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
                          ("bin", "passthrough", ["senior"])], sparse_threshold=0)
### START CODE HERE ### (≈ 30 dòng)
# STARTER: # auc_lr: make_pipeline(pre9, LogisticRegression(max_iter=2000)) → roc_auc_score trên test
# STARTER: # MLP: A_tr = pre9.fit_transform(X9_tr) (float32); tách 20% làm validation; nn.Sequential(Linear, ReLU, Dropout?, Linear(…, 1))
# STARTER: # BCEWithLogitsLoss, Adam(lr=1e-3), mini-batch 64; mỗi epoch đo AUC validation, giữ state_dict tốt nhất; patience 10
# STARTER: auc_lr, auc_mlp = None, None
auc_lr = roc_auc_score(y9_te, make_pipeline(pre9, LogisticRegression(max_iter=2000)).fit(X9_tr, y9_tr).predict_proba(X9_te)[:, 1])
A_tr_all = pre9.fit_transform(X9_tr).astype(np.float32)
A_te = pre9.transform(X9_te).astype(np.float32)
A_tr, A_va, t_tr, t_va = train_test_split(A_tr_all, y9_tr, test_size=0.2, random_state=0, stratify=y9_tr)
torch.manual_seed(0)
mlp9 = nn.Sequential(nn.Linear(A_tr.shape[1], 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1))
opt9 = torch.optim.Adam(mlp9.parameters(), lr=1e-3)
loss_fn9 = nn.BCEWithLogitsLoss()
Xa, ya = torch.tensor(A_tr), torch.tensor(t_tr, dtype=torch.float32)
Xv = torch.tensor(A_va)
best_auc, best_state, bad = -1.0, None, 0
for epoch in range(200):
    mlp9.train()
    perm = torch.randperm(len(Xa))
    for s in range(0, len(perm), 64):
        idx = perm[s:s + 64]
        opt9.zero_grad()
        loss_fn9(mlp9(Xa[idx]).squeeze(1), ya[idx]).backward()
        opt9.step()
    mlp9.eval()
    with torch.no_grad():
        auc_va = roc_auc_score(t_va, mlp9(Xv).squeeze(1).numpy())
    if auc_va > best_auc:
        best_auc, best_state, bad = auc_va, {k: v.clone() for k, v in mlp9.state_dict().items()}, 0
    else:
        bad += 1
        if bad >= 10:
            break
mlp9.load_state_dict(best_state)
mlp9.eval()
with torch.no_grad():
    auc_mlp = roc_auc_score(y9_te, mlp9(torch.tensor(A_te)).squeeze(1).numpy())
### END CODE HERE ###
print(f"ROC-AUC test: Logistic Regression = {auc_lr:.4f} | MLP = {auc_mlp:.4f}")

In [ ]:
check_greater(auc_mlp, auc_lr - 0.01, "MLP không tệ hơn Logistic Regression quá 0.01")
check_greater(auc_lr, 0.75, "baseline hợp lý")

**Nhận xét mẫu:** Trên dữ liệu bảng vài nghìn dòng, ít feature, quan hệ gần tuyến tính trong không gian logit, MLP chỉ **ngang** Logistic Regression — nhưng tốn công hơn nhiều (tiền xử lý, early stopping, seed, siêu tham số) và khó diễn giải. Với dữ liệu bảng, gradient boosting (chương 12) thường là lựa chọn mạnh nhất; deep learning tỏa sáng với dữ liệu **phi cấu trúc** (ảnh, văn bản, âm thanh) và dữ liệu rất lớn, nơi biểu diễn học được có giá trị (chương 29–34). Luôn so với baseline đơn giản.

## L7 — Nâng cao

**Bài 10.** **Autograd cho tensor.** Mở rộng ý tưởng engine ở bài 02 từ số vô hướng sang **mảng NumPy**: lớp `Tensor` với `data` (ndarray), `grad` (cùng shape), và các phép `__add__` (hỗ trợ broadcast bias dạng $(n, m) + (m,)$ — gradient của bias là **tổng theo trục bị broadcast**), `__sub__`, `__matmul__`, `relu()`, `__pow__` (số mũ vô hướng), `mean()` (trả về tensor vô hướng); `backward()` theo thứ tự tô-pô. Kiểm tra gradient của mạng 2 lớp với loss MSE so với PyTorch; dùng nó huấn luyện hồi quy $y = \sin(3x)$.

In [ ]:
class Tensor:
    def __init__(self, data, parents=()):
        self.data = np.asarray(data, float)
        self.grad = np.zeros_like(self.data)
        self._parents = parents
        self._backward = lambda: None

    ### START CODE HERE ### (≈ 50 dòng)
    # STARTER: # __add__: out = data + other.data; backward: self.grad += _unbroadcast(out.grad, self.data.shape), tương tự other
    # STARTER: # __matmul__: backward: self.grad += out.grad @ other.data.T; other.grad += self.data.T @ out.grad
    # STARTER: # relu, __pow__(k), mean(), __neg__, __sub__, backward() theo thứ tự tô-pô (grad của nút cuối = 1)
    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, (self, other))
        def _backward():
            self.grad += _unbroadcast(out.grad, self.data.shape)
            other.grad += _unbroadcast(out.grad, other.data.shape)
        out._backward = _backward
        return out

    def __neg__(self):
        out = Tensor(-self.data, (self,))
        def _backward():
            self.grad += -out.grad
        out._backward = _backward
        return out

    def __sub__(self, other):
        return self + (-(other if isinstance(other, Tensor) else Tensor(other)))

    def __matmul__(self, other):
        out = Tensor(self.data @ other.data, (self, other))
        def _backward():
            self.grad += out.grad @ other.data.T
            other.grad += self.data.T @ out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Tensor(np.maximum(0, self.data), (self,))
        def _backward():
            self.grad += out.grad * (self.data > 0)
        out._backward = _backward
        return out

    def __pow__(self, k):
        out = Tensor(self.data**k, (self,))
        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def mean(self):
        out = Tensor(self.data.mean(), (self,))
        def _backward():
            self.grad += np.ones_like(self.data) * out.grad / self.data.size
        out._backward = _backward
        return out

    def backward(self):
        order, visited = [], set()
        def build(t):
            if id(t) not in visited:
                visited.add(id(t))
                for p in t._parents:
                    build(p)
                order.append(t)
        build(self)
        self.grad = np.ones_like(self.data)
        for t in reversed(order):
            t._backward()
    ### END CODE HERE ###


def _unbroadcast(grad, shape):
    """Cộng gradient theo các trục đã bị broadcast để về đúng shape gốc."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for ax, s in enumerate(shape):
        if s == 1:
            grad = grad.sum(axis=ax, keepdims=True)
    return grad

In [ ]:
r10 = np.random.default_rng(3)
Xd, Yd = r10.normal(size=(16, 3)), r10.normal(size=(16, 1))
params_np = [r10.normal(size=(3, 5)), r10.normal(size=5), r10.normal(size=(5, 1)), r10.normal(size=1)]
W1t, b1t, W2t, b2t = (Tensor(p.copy()) for p in params_np)
loss10 = (((Tensor(Xd) @ W1t + b1t).relu() @ W2t + b2t - Tensor(Yd)) ** 2).mean()
loss10.backward()
tp = [torch.tensor(p, requires_grad=True) for p in params_np]
tloss10 = (((torch.tensor(Xd) @ tp[0] + tp[1]).relu() @ tp[2] + tp[3] - torch.tensor(Yd)) ** 2).mean()
tloss10.backward()
check_close(loss10.data, tloss10.item(), tol=1e-10, message="loss khớp PyTorch")
for name, mine, ref in zip(["W1", "b1", "W2", "b2"], [W1t, b1t, W2t, b2t], tp):
    check_close(mine.grad, ref.grad.numpy(), tol=1e-10, message=f"grad {name} khớp PyTorch (kể cả bias broadcast)")

In [ ]:
xs10 = np.linspace(-1, 1, 128)[:, None]
ys10 = np.sin(3 * xs10)
r = np.random.default_rng(0)
P10 = [Tensor(r.normal(0, 1.0, (1, 32))), Tensor(np.zeros(32)), Tensor(r.normal(0, np.sqrt(2 / 32), (32, 1))), Tensor(np.zeros(1))]
losses10 = []
for step in range(2000):
    for p in P10:
        p.grad = np.zeros_like(p.data)
    loss = (((Tensor(xs10) @ P10[0] + P10[1]).relu() @ P10[2] + P10[3] - Tensor(ys10)) ** 2).mean()
    loss.backward()
    for p in P10:
        p.data -= 0.05 * p.grad
    losses10.append(float(loss.data))
print(f"MSE: {losses10[0]:.4f} → {losses10[-1]:.4f}")
check_less(losses10[-1], 0.02, "học được sin(3x)")

**Nhận xét:** Chuyển từ vô hướng sang tensor chỉ đổi **công thức gradient cục bộ** của từng phép (ma trận chuyển vị, cộng theo trục broadcast) — cấu trúc đồ thị và thứ tự tô-pô giữ nguyên. Đây là cốt lõi của PyTorch/JAX; phần còn lại (hàng trăm phép toán, GPU kernel, quản lý bộ nhớ, `no_grad`, hooks) là kỹ thuật. Lỗi kinh điển khi tự viết: quên `_unbroadcast` cho bias → shape gradient sai hoặc gradient bias sai.